In [ ]:
pip install OrderFusion

# Tutorial

In [ ]:
import OrderFusion

country = 'austria' # options: 'germany' or 'austria'
resolution = 'h' # options: 'h' for 60-min products, 'qh' for 15-min products
train_start, train_end = '2022-01-01', '2022-01-15'
val_start,   val_end   = '2022-01-15', '2022-02-01'
test_start,  test_end  = '2022-02-01', '2022-03-01'

#train_start, train_end = '2022-01-01', '2024-01-01'
#val_start,   val_end   = '2024-01-01', '2024-07-01'
#test_start,  test_end  = '2024-07-01', '2025-01-01'

indice = 'ID1' # options: 'ID1', 'ID2', 'ID3'

'''
only obtain most recent "num_trade" trades for data efficiency; 
could set to arbitrarily large value to use all trades,
but should larger than cutoff_len.
'''
num_trade = 128 

'''
assign the temporal weight of 1 to the most recent "cutoff_len" trades; 
distant trades are assigned weight of 0.
'''
cutoff_len = 16

'''
"pad_value" is used to fill the missing values when the number of trades is less than "num_trade". 
It should be outside the normal range of orderbook prices i.e. [-∞, -9999] or [9999, ∞].
'''
pad_value = 10000.0 

'''
target quantiles to forecast, could be many e.g. [0.05, 0.1, 0.5, 0.9, 0.95]
'''
quantiles = [0.1, 0.5, 0.9] 

hidden_dim, max_degree, num_heads, epoch, batch_size, seed = 16, 2, 1, 50, 256, 42
model_mode = 'OrderFusion'
show_progress_bar = True # options: True, False
save_path = OrderFusion.device_choice('local') # 'local' if run locally or 'cloud' run on Google Colab

# meta setting
setting = (country, resolution, indice, hidden_dim, max_degree, num_heads, epoch, batch_size, save_path, model_mode, [int(q * 100) for q in quantiles], cutoff_len, show_progress_bar)

c:\Users\YuR\AppData\Local\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Step 1: extract input and output from orderbook

Separate 60-min and 15-min orderbook

In [2]:
OrderFusion.filter_data(country, train_start, test_end) # --> CSV files of 60-min and 15-min products are separately saved in the 'data' folder

  processing austria 2022-01-01 → 2022-03-01: 100%|██████████| 59/59 [00:47<00:00,  1.25file/s]


Merge files per resolution and country

In [3]:
OrderFusion.merge_data(resolution, country)  # --> data are merged per resolution and country

Merging files:
  - raw_2022_01_h_austria.pkl
  - raw_2022_02_h_austria.pkl


Extract input sequences per side

In [4]:
OrderFusion.get_input(resolution, country, indice)  # --> extracted sequences for buy side and sell side, respectively, are saved in the 'data' folder

  Extracting sequences: 100%|██████████| 1415/1415 [00:04<00:00, 341.79group/s, Processing date: 2022-02-28 22:00:00+00:00]


Extract output label (ID1, ID2, ID3)

In [5]:
OrderFusion.get_output(resolution, country, indice)  # --> extracted labels (ID1, ID2, ID3) are saved in the 'data' folder

  Extracting labels: 100%|██████████| 1415/1415 [00:00<00:00, 1481.60group/s]


Fit data scaler and save

In [6]:
OrderFusion.get_scaler(country, resolution, train_start, train_end)  # --> fitted scalers are saved in the 'data' folder

  Scaler saved to Data/scaler_austria_h.pkl


# Step 2: read and preprocess input and output

In [7]:
save_path = OrderFusion.device_choice('local') 
orderbook_df = OrderFusion.read_data(save_path, country, resolution, indice)
orderbook_df.head()

,Sequence_Buy,SumVolume_x,NumTrades_x,Sequence_Sell,SumVolume_y,NumTrades_y,ID1,UTC
0,"[[1.0, 2.0, 32337.155], [1.0, 2.0, 32336.842],...",274.0,107.0,"[[1.0, 2.0, 32337.367], [1.0, 2.0, 32337.155],...",176.1,69.0,25.531999,2021-12-31 23:00:00+00:00
1,"[[1.0, 2.0, 35937.091], [1.0, 2.0, 35757.499],...",156.8,75.0,"[[1.0, 2.0, 35937.367], [1.0, 2.0, 35937.091],...",206.9,81.0,17.046015,2022-01-01 00:00:00+00:00
2,"[[1.0, 2.0, 39527.345], [1.0, 2.0, 39527.232],...",120.0,59.0,"[[1.0, 2.0, 39537.367], [1.0, 2.0, 39527.345],...",321.0,167.0,15.993121,2022-01-01 01:00:00+00:00
3,"[[1.0, 2.0, 43137.023], [1.0, 10.0, 42970.141]...",163.0,73.0,"[[1.0, 2.0, 43137.023], [1.0, 10.0, 42970.141]...",342.6,165.0,32.691250,2022-01-01 02:00:00+00:00
4,"[[1.0, 2.0, 46737.01], [1.0, 10.0, 46575.347],...",181.8,88.0,"[[1.0, 2.0, 46737.01], [1.0, 10.0, 46575.347],...",303.9,123.0,3.067633,2022-01-01 03:00:00+00:00


Split data intro training, validation, and testing

In [19]:
X_train, y_train, X_val, y_val, X_test, y_test = OrderFusion.split_data(orderbook_df, indice, train_start, train_end, val_start, val_end, test_start, test_end)

Scale data for each split

In [20]:
X_train, y_train, X_val, y_val, X_test, y_test = OrderFusion.scale_data(X_train, y_train, X_val, y_val, X_test, y_test, save_path, country, resolution)


Pad data 

In [21]:
X_train, X_val, X_test = OrderFusion.pad_data(X_train, X_val, X_test, num_trade, pad_value)

# Step 3: train and optimize model

In [ ]:
best_model, hist, paras_count = OrderFusion.optimize_model(X_train, y_train, X_val, y_val, setting)

# Step 4: evaluate model's performance

In [23]:
OrderFusion.evaluate_model(best_model, X_test, y_test, quantiles, save_path, country, resolution)

138/138 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step
AQL: 6.12, AQCR: 0.0, RMSE: 39.55, MAE: 17.95, R2: 0.69, Inference time: 2.979466676712036s 



# Step 5: inference 

In [ ]:
best_model = OrderFusion.load_best_model(quantiles, country, resolution, indice)
y_pred, y_test = OrderFusion.get_forecasts(best_model, save_path, X_test, y_test, quantiles, country, resolution)

138/138 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step


# Optional: plotting forecasts versus true prices

In [ ]:
OrderFusion.plot_forecasts(y_pred, y_test, 181, 400, indice)
OrderFusion.gif_conversion(indice)